# CI/CD & Automated Quality Gates

## Objective

The project now contains a data warehouse, dbt transformations, machine-learning models, decision logic, an API, Docker packaging, Airflow orchestration, and incremental-ingestion controls.

Until this point, I manually ran tests and validation commands before committing changes.

In this notebook, I introduce continuous integration using GitHub Actions.

The CI workflow creates a clean environment after code changes, installs the project dependencies, checks repository hygiene, validates Python syntax, runs the full unit-test suite, parses the dbt project, and builds the FastAPI Docker image.

The objective is to detect integration problems automatically rather than relying on a developer to remember every validation step.

The workflow intentionally does not access Snowflake, AWS, MLflow runtime state, or production credentials. Cloud integration and deployment remain separate controlled processes.

## 1. Continuous integration architecture

Continuous integration validates code whenever the repository changes.

The workflow begins with repository checkout on a temporary GitHub-hosted runner. It then reconstructs the project's Python environment from `requirements.txt` and executes a shared quality-gate script.

The quality gates cover four areas:

1. Repository hygiene — generated datasets, model artifacts, credentials, and local environments must not be tracked by Git.
2. Python syntax — application, pipeline, script, and DAG files must parse successfully.
3. Unit testing — deterministic tests verify decision logic, API behavior, pipeline checks, and incremental-ingestion contracts.
4. dbt parsing — the transformation graph must compile using a temporary nonproduction profile without connecting to Snowflake.

Only after those checks pass does a second job build the FastAPI Docker image.

The workflow builds but does not publish or deploy the image. Deployment remains separate from continuous integration.

In [ ]:
# ============================================================
# Run the same quality gates used by GitHub Actions
#
# This allows me to reproduce CI failures locally before
# pushing another commit.
#
# The source of truth remains scripts/ci_checks.py.
# The notebook does not duplicate the validation logic.
# ============================================================

from pathlib import Path
import subprocess
import sys


PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (
        path
        / "dbt"
        / "dbt_project.yml"
    ).exists()
)


result = subprocess.run(
    [
        sys.executable,
        "scripts/ci_checks.py",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)


print(result.stdout)


if result.returncode != 0:

    print(result.stderr)

    raise RuntimeError(
        "Local CI quality gates failed."
    )


print(
    "\nPASS: Local quality gates "
    "match the CI validation workflow."
)

## 2. What a successful CI run proves

A successful CI run shows that the repository can be checked on a clean machine using its declared dependencies.

It verifies that the offline unit tests pass, the dbt project parses, repository-artifact rules are respected, and the API Docker image can be built.

A successful CI run does **not** establish that:

- Snowflake is currently available.
- S3 permissions are valid.
- The Airflow DAG will successfully execute against live infrastructure.
- The deployed model has good causal performance.
- A newly trained model should replace the currently served model.
- The Docker image is ready for public production deployment.

Those concerns require integration testing, model evaluation, infrastructure controls, and deployment policies beyond this CI workflow.

## Conclusions and limitations

I introduced continuous integration using GitHub Actions to automatically validate repository changes on a clean environment.

The workflow reconstructs the Python environment from the repository's declared dependencies and applies shared quality gates covering repository hygiene, Python syntax, unit tests, and dbt parsing.

A second CI job builds the FastAPI Docker image only after the quality checks succeed.

### Key findings

- CI platform: **GitHub Actions**
- Python runtime: **3.10**
- Repository hygiene check: **Passed**
- Python syntax check: **Passed**
- Unit tests: **Passed**
- dbt static parse: **Passed**
- Docker image build: **Passed**
- Overall CI status: **Success**

### Reliability improvements

The workflow prevents several classes of repository errors from going unnoticed, including failed unit tests, broken Python syntax, invalid dbt project structure, broken Docker builds, and accidental tracking of generated or private project artifacts.

The same quality-gate script can be executed locally before a commit and remotely by GitHub Actions, reducing differences between local and CI validation logic.

### Limitations

This workflow performs offline continuous integration rather than full infrastructure integration testing.

It does not connect to Snowflake or AWS, execute the Airflow DAG, retrain models, publish Docker images, or deploy the API.

The Docker job verifies that the serving image can be constructed, but it does not run the real model artifact inside the container because deployment artifacts are intentionally excluded from Git.

A production software-delivery process would add protected branches, pull-request review, dependency and security scanning, artifact registries, environment-specific deployment controls, and post-deployment monitoring.